# 17장 · 도구 호출 루프와 종료 조건

도구 호출 루프는 모델이 다음 도구를 요청할 때마다 실행 결과를 기록하고 다시 호출합니다. 최종 답변과 호출 상한은 서로 다른 종료 사유입니다.

주말 운영 시간과 대출 기간을 서로 다른 함수에서 조회합니다. 각 도구의 결과를 메시지 이력에 추가하고, 남은 호출 횟수와 모델 응답에 따라 반복을 종료합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 도구 결과를 대화 기록에 누적

두 도구를 쓰기 위해 이 장에서는 이름을 get_hours와 get_loan_rule로 정했습니다. 함수 이름이 달라도 각 요청과 결과를 같은 call_id로 연결한다는 원칙은 16장과 같습니다.

### 셀 01 · 첫 function_call 실행

제공된 execute는 이름·인자를 검사하고 조회 값을 JSON으로 묶습니다. 하나의 도구 결과부터 읽으세요.

In [1]:
import json

def execute(item):
    try:
        args = json.loads(item["arguments"])
        if not isinstance(args, dict):
            raise ValueError("객체 필요")
        if item["name"] == "get_hours" and set(args) == {"day"}:
            hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}
            result = {"ok": True, "hours": hours[args["day"]]}
        elif item["name"] == "get_loan_rule" and args == {}:
            result = {"ok": True, "loan_days": 14}
        else:
            raise ValueError("허용하지 않은 요청")
    except (ValueError, TypeError, KeyError):
        result = {"ok": False, "error": "invalid_tool_request"}
    return {"type": "function_call_output", "call_id": item["call_id"],
            "output": json.dumps(result, ensure_ascii=False)}

first = {"type": "function_call", "name": "get_hours", "call_id": "c1", "arguments": '{"day":"weekend"}'}
first_result = execute(first)
print(json.loads(first_result["output"])["hours"])

10:00~16:00


c1 요청에서 찾은 10:00~16:00입니다. 다음 요청은 대출 기간을 묻기 때문에 새 식별자 c2를 써야 합니다.

원래 질문은 주말 운영 시간과 대출 기간을 함께 요구합니다. 첫 도구 결과에는 10:00~16:00만 있습니다. 이 시점에 루프를 끝내면 대출 기간은 비어 있습니다. 실제 에이전트에서는 모델이 다음 호출을 결정합니다. 이 실습에서는 미리 정해 둔 두 번째 요청 c2로 대출 기간을 조회합니다.

### 서로 다른 두 도구의 인자

<p>get_hours는 day가 weekday 또는 weekend인지 확인한 후 hours를 반환합니다. get_loan_rule은 빈 인자 객체를 받아 loan_days에 14를 넣습니다. 한 도구의 인자 규칙을 다른 도구에 그대로 적용할 수 없습니다.</p>

### 셀 02 · 연속 도구 호출과 대화 이력

먼저 처리한 c1의 요청·응답 뒤에 c2의 요청과 결과를 넣습니다.

In [2]:
history = [{"role": "user", "content": "주말 시간과 대출 기간은?"}, first, first_result]
second = {"type": "function_call", "name": "get_loan_rule", "call_id": "c2", "arguments": "{}"}
history.append(second)
history.append(execute(second))
print(json.loads(history[-1]["output"])["loan_days"])

14


마지막 도구에서 반환한 값은 14일입니다. 아직 최종 안내는 기록에 없습니다. 모델은 두 결과를 본 뒤 답을 쓰기 위해 한 번 더 호출돼야 합니다.

### 도구 요청과 결과가 쌓이는 이력

<p>history의 첫 항목은 사용자 질문입니다. 그 뒤에는 c1의 호출과 결과가 한 쌍으로 들어 있고, 이어서 c2의 호출과 결과가 추가됩니다. 마지막 결과 안쪽의 loan_days를 읽으면 14입니다.</p>

## 최종 응답과 최대 호출 횟수

모델 호출과 도구 실행은 따로 셉니다. 이번 대본의 첫 두 응답은 도구 호출이고 세 번째 응답은 답변입니다. 아래 함수는 진짜 모델 대신 준비된 응답을 순서대로 읽는 작은 루프입니다. 잘못된 ID와 중복을 검사하는 전체 수업 예제는 같은 종료 규칙을 더 엄격하게 적용합니다.

### 도구를 실행한 뒤의 다음 응답

<p>주말 시간과 대출 기간을 함께 묻는 질문에 첫 응답은 운영 시간, 둘째 응답은 대출 규칙을 요청합니다. 이 장은 세 모델 응답을 script에 미리 담아 두고, 어떤 입력에서 history가 늘어나거나 루프가 끝나는지 관찰합니다.</p>

### 요청 c2가 이력에 있어도 실행 결과가 없는 이유

max_calls를 2로 낮추면 두 번째 모델 응답을 기록한 뒤 상한에 걸립니다. 마지막 이력 항목은 c2의 호출 요청입니다.

1. 모델 응답 · 두 번째 응답 = get_loan_rule
2. 이력 추가 · history의 마지막 ID = c2
3. 예산 확인 · number / max_calls = 2 / 2
4. 상한 종료 · status = model_limit

### 셀 03 · 최종 메시지에서 루프 종료

회차마다 먼저 응답 종류를 읽습니다. 최종 응답을 받거나 상한에 닿으면 함수를 빠져나갑니다.

모델이 더 요청할 도구가 없다고 답하면 이 루프는 끝납니다. 요청 ID 중복을 막으려면 execute(item)를 부르기 전에 이미 처리한 ID인지 확인해야 합니다. 실행 뒤에 ID를 비교하면 같은 도구가 이미 두 번 작동했을 수 있습니다.

In [3]:
def run_script(items, max_calls):
    history = [{"role": "user", "content": "주말 시간과 대출 기간은?"}]
    for number, item in enumerate(items, start=1):
        if number > max_calls:
            break
        history.append(item)
        if item["type"] == "message":
            return {"status": "completed", "history": history, "model_calls": number}
        if number == max_calls:
            return {"status": "model_limit", "history": history, "model_calls": number}
        history.append(execute(item))
    return {"status": "incomplete", "history": history, "model_calls": min(len(items), max_calls)}

answer = {"type": "message", "text": "주말 운영 시간은 10:00~16:00이고 대출 기간은 14일입니다."}
script = [first, second, answer]
trace = run_script(script, max_calls=3)
print(trace["model_calls"], trace["history"][-1]["text"])

3 주말 운영 시간은 10:00~16:00이고 대출 기간은 14일입니다.


세 번째 응답은 message이므로 반복을 끝냅니다. 응답 목록이 먼저 바닥나면 함수 마지막 줄에서 incomplete를 반환합니다. 외부 모델을 부르지 않고도 두 종료 경로를 따로 재현할 수 있습니다.

### 응답 종류와 호출 상한 검사 순서

<p>run_script는 응답 항목을 history에 넣은 뒤 type을 확인합니다. message이면 completed로 끝납니다. 도구 요청이고 이미 max_calls에 도달했다면 model_limit을 반환합니다. 예산이 남을 때만 도구를 실행해 결과를 history에 붙입니다.</p>

### 복수 도구 실행 로그

<p>script의 각 항목은 미리 작성한 응답입니다. 반면 function_call_output은 execute를 실행한 자리에서 만들어집니다. history에서 두 종류를 구분하면 도구를 실제로 몇 번 실행했는지 셀 수 있습니다.</p>

### 과제 04 · max_calls 초과 판정

최대 두 번까지만 모델을 부를 수 있게 바꿔 종료 이유와 history 마지막 항목의 ID를 출력하세요.

In [4]:
# max_calls를 줄여 trace를 다시 얻으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>limited = run_script(script, max_calls=2)
print(limited[&quot;status&quot;], limited[&quot;history&quot;][-1][&quot;call_id&quot;])</code></pre>
<p>실행 결과</p><pre><code>model_limit c2</code></pre>
<p>model_limit c2입니다. c2 요청은 받았지만 실행 뒤 답을 받을 기회가 없으므로 둘째 도구 실행 전에 멈췄습니다. c2의 결과가 이미 있다고 해석하면 안 됩니다.</p>
</details>

## call_id 중복 검사

같은 요청이 두 번 들어왔다고 해서 같은 작업을 다시 실행해도 된다는 뜻은 아닙니다. 실행한 호출의 ID를 모아서 새 요청을 허용할지 먼저 판단합니다.

### 과제 05 · 집합으로 호출 ID 중복 검사

이미 실행한 c1과 같은 ID가 다시 왔습니다. 실행 전 중복 여부를 표시하세요.

In [5]:
seen = {"c1"}
# first가 이미 실행된 요청인지 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>seen = {&quot;c1&quot;}
print(first[&quot;call_id&quot;] in seen)</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True이므로 같은 ID를 다시 실행하지 않습니다. 종료 상태를 남기면 요청 자체가 없던 경우와 구분해 원인을 찾을 수 있습니다.</p>
</details>

### 같은 호출 ID를 다시 받았을 때

<p>seen에 c1을 넣고 첫 요청의 ID를 대조하면 True가 나옵니다. 이 셀은 중복 검사 한 가지를 따로 실행합니다. run_script에서도 같은 요청의 재실행을 막으려면 도구를 부르기 전에 이 검사를 연결해야 합니다.</p>

## 돌아보기

대화 기록에서 c2가 마지막일 때와 c2의 결과가 마지막일 때를 비교해 보세요. 같은 ID가 보이더라도 하나는 요청, 하나는 실행 뒤 기록입니다. 호출 상한을 두는 위치에 따라 실제 실행 횟수가 달라집니다.

### 과제 06 · 루프에 중복 실행 방지 넣기

도구 호출 직전에 이미 처리한 ID를 검사하는 run_once_per_id(items)를 작성하세요. 첫 요청이 재전송되면 이력을 거기서 멈추고, 실제로 만들어진 결과의 ID만 출력합니다.

In [ ]:
replayed = [first, first, second]
# 중복 요청에서 멈추고, 실행한 결과의 ID 목록을 출력하세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>replayed = [first, first, second]

def run_once_per_id(items):
    results = {}
    for request in items:
        call_id = request["call_id"]
        if call_id in results:
            break
        results[call_id] = execute(request)
    return list(results)

print(run_once_per_id(replayed))</code></pre>
<p>실행 결과</p><pre><code>['c1']</code></pre>
<p>두 번째 c1은 이미 저장한 결과와 연결됩니다. break 뒤의 c2에도 결과가 없습니다. 이 예제의 기록은 함수 호출이 끝나면 사라집니다.</p>
</details>